# Изучение набора данных


> 🚧 Раздел в разработке.


Перед анализом важно “познакомиться” с данными: понять, **что хранится в таблице**, **какая геометрия**, **в какой системе координат**, **есть ли пропуски и ошибки**, и **какие атрибуты доступны для расчётов**.

На примере одного из наборов пространственных данных в формате **GeoDataFrame**, мы посмотрим, как это можно сделать


## 0. Импорт библиотек


In [ ]:
import geopandas as gpd

- [**GeoPandas**](https://geopandas.org/) (`geopandas`) — расширение библиотеки pandas, предназначенное для работы с геопространственными данными. Позволяет загружать, обрабатывать и анализировать пространственные наборы данных в различных форматах.


## 1. Основная информация


Прочитаем данные из формата `.geojson`


In [ ]:
gdf_geojson = gpd.read_file("data/spb_metro.geojson") 

### 1.1 `head()`

Метод `head()` отображает первые строки `DataFrame` или `GeoDataFrame`.
Он используется для первичного знакомства с содержимым данных.

По умолчанию выводятся первые **5 строк**.


In [ ]:
gdf_geojson.head()

Можно указать количество строк вручную:


In [ ]:
gdf_geojson.head(10)

### 1.2 `info()`

Метод `info()` отображает краткую сводную информацию о `DataFrame` или `GeoDataFrame`:

- количество записей (строк),
- названия столбцов,
- типы данных в каждом столбце,
- количество **непустых значений**,
- объём памяти, занимаемый данными.


In [ ]:
gdf_geojson.info()

### 1.3 `len()` и `shape`

Чтобы понять размер набора данных, удобно использовать метод `len()` и атрибут `shape`.

Функция `len()` возвращает количество строк в `DataFrame` или `GeoDataFrame`,
то есть **количество пространственных объектов**.


In [ ]:
len(gdf_geojson)

Атрибут `shape` возвращает кортеж из двух чисел:

- первое число — количество строк (объектов),
- второе число — количество столбцов (атрибутов, включая `geometry`).


In [ ]:
gdf_geojson.shape

## 2. Геометрия


В `GeoDataFrame` каждый объект имеет **геометрию**

Перед выполнением пространственного анализа важно ответить на несколько вопросов:

- Какие типы геометрии содержатся в данных (точки, линии, полигоны)?
- Корректны ли геометрии?


### 2.2 `geometry`

В `GeoDataFrame` пространственная информация хранится в специальном столбце `geometry`.
Именно он отличает `GeoDataFrame` от обычного `DataFrame`.


Этот атрибут возвращает объект типа `GeoSeries`, содержащий геометрию каждого пространственного объекта.


In [ ]:
gdf_geojson.geometry

Иногда в наборе данных встречаются объекты без геометрии.
Это критично, потому что такие строки не могут участвовать в пространственном анализе.

При работе с пространственными данными важно различать:

- `NaN` — геометрия отсутствует,
- `empty geometry` — геометрия формально существует, но она пустая.


#### 2.1.1 Проверка пропусков (`NaN`)

Метод:

- `isna()` — определяет пропущенные значения,
- `sum()` — подсчитывает их количество.


In [ ]:
gdf_geojson.geometry.isna().sum()

Если значение больше нуля — часть строк содержит `NaN`, и их необходимо обработать (например, удалить или восстановить геометрию).

Удалить можно, например, так:


In [ ]:
gdf_geojson = gdf_geojson.dropna(subset=["geometry"])

#### 2.1.2 Проверка пустых геометрий


In [ ]:
gdf_geojson.geometry.is_empty.sum()

#### 2.1.3 Поле, где записана геометрия

В `GeoDataFrame` всегда есть активное поле геометрии — именно он используется для всех пространственных операций.

По умолчанию его имя — `geometry`, но оно может быть другим (например, после переименования или создания новой геометрии).


In [ ]:
gdf_geojson.geometry.name

- В `GeoDataFrame` может быть несколько столбцов с геометриями, но только один из них является активным.
- Все пространственные операции (`buffer`, `intersection`, `area`, `plot`) выполняются именно над активным столбцом геометрии.


### 2.2. `geom_type`


Чтобы определить тип геометрии объектов, используется атрибут `geom_type`. Он возвращает серию (`GeoSeries`), в которой для каждой строки указан тип геометрии:


gdf_geojson.geom_type.unique()


Посмотрим, какие уникальные типы встречаются в данных


In [ ]:
gdf_geojson.geom_type.value_counts()

### 2.2. `is_valid`


Помимо пропусков и пустых объектов, геометрия может быть **невалидной** (геометрически некорректной).

Чаще всего проблемы с некорректной геометрией возникают у полигонов, например:

- самопересечение (self-intersection),
- незамкнутые контуры,
- дублирующиеся вершины,
- некорректная топология.


Можно проверить количество некорректных геометрий в наборе данных:

- метод `.is_valid` — возвращает `True` или `False` для каждой геометрии;
- `~` — логическое отрицание (находит невалидные объекты);
- `sum()` — подсчитывает их количество.


In [ ]:
(~gdf_geojson.is_valid).sum()

Посмотреть, какие именно объекты имеют некорректную геометрию


In [ ]:
gdf_geojson[~gdf_geojson.is_valid]

Исправить некорректную геометрию можно так


In [ ]:
gdf_geojson["geometry"] = gdf_geojson.make_valid()

После исправления стоит повторно проверить


In [ ]:
(~gdf_geojson.is_valid).sum()

## 3. Система координат (CRS)


Координаты для геометрий в `GeoDataFrame` записаны в определённой **системе координат (CRS — Coordinate Reference System)**.

Перед выполнением пространственного анализа важно ответить на несколько вопросов:

- Указана ли система координат в данных?
- Какой у неё тип?
- Подходит ли текущая CRS для расчёта расстояний и площадей?
- Совпадает ли CRS у всех используемых слоёв?

Здесь мы рассмотрим основные характеристики CRS, которые можно определить на этапе первичного анализа. Подробнее о системах координат поговорим во втором модуле.


### 3.1 `crs`


Атрибут `crs` позволяет узнать, в какой системе координат записаны пространственные данные.


In [ ]:
gdf_geojson.crs

CRS (Coordinate Reference System) определяет:

- систему отсчёта координат,
- единицы измерения (градусы или метры),
- способ проецирования поверхности Земли на плоскость.


### 3.2 Тип системы координат


Система координат может быть:

- **географической** — координаты заданы в градусах (широта и долгота),
- **проекционной** — координаты заданы в линейных единицах (чаще всего в метрах).

Проверить тип CRS можно так:


In [ ]:
print("Географическая:", gdf_geojson.crs.is_geographic)
print("Проекционная:", gdf_geojson.crs.is_projected)

- Если `is_geographic == True` — координаты в градусах (например, `EPSG:4326`).
- Если `is_projected == True` — координаты в метрах или других линейных единицах.


### 3.3 Единицы измерения координат


Можно узнать, в каких единицах записаны координаты:

- `degree` — градусы,
- `metre` — метры
  могут быть указаны и другие единицы измерения


In [ ]:
gdf_geojson.crs.axis_info

- `degree` — градусы,
- `metre` — метры


### 3.4 Получение EPSG-кода


Иногда удобно получить только числовой код системы координат:


In [ ]:
gdf_geojson.crs.to_epsg()

Если CRS нестандартная, метод может вернуть `None`.


## 4. Атрибутивная информация


Помимо геометрии, каждый объект в `GeoDataFrame` содержит **атрибутивную информацию** — описательные характеристики, связанные с пространственными объектами.

В этом разделе мы рассмотрим, как изучать структуру атрибутивных данных, проверять типы полей, искать пропуски и получать базовую статистику.


### 4.1 `columns`

Атрибут `columns` позволяет получить список всех столбцов в `DataFrame` или `GeoDataFrame`.
Результат представляет собой объект типа `Index`, содержащий названия всех полей, включая поле с геометрией:


In [ ]:
gdf_geojson.columns

### 4.2 `dtypes`

Атрибут `dtypes` позволяет узнать тип данных каждого столбца в `DataFrame` или `GeoDataFrame`.


In [ ]:
gdf_geojson.dtypes

Типы данных необходимо проверять, чтобы убедиться, что каждое поле корректный формат и с ним можно выполнять соответствующие аналитические операции без ошибок


## 5. Другие методы изучения

Рассмотрим методы, которые не относятся к категориям выше.


### 5.1 Границы набора данных

**Bounding Box** — это минимальный прямоугольник, который полностью охватывает все объекты набора данных.

Он задаётся четырьмя координатами:

```
[minx, miny, maxx, maxy]
```

Получить границы всего набора данных можно так:


In [ ]:
gdf_geojson.total_bounds

## 6. Итог


В этом разделе мы познакомились с основными способами первичного анализа пространственного набора данных в формате `GeoDataFrame`.

Это очень важный этап проверк загруженных данных, так как позволяет понять структуру и содержание данных, убедиться в корректности загрузки, выявить потенциальные ошибки
